# Opisy canonical values — słownik pojęć katalogu

Dla każdej canonical value z `zurada_canonical_mappings.json` generujemy krótki opis po polsku.  
Wynik: `zurada_canonical_descriptions.csv` z kolumnami `value`, `column`, `description`.

Przetwarzamy batchami (20 wartości / request) → ~18 wywołań API łącznie.

In [1]:
%pip install openai pandas tqdm --quiet

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os, json, time
import pandas as pd
from tqdm.notebook import tqdm
from openai import OpenAI

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "Brak klucza API")  # ustaw zmienną środowiskową lub wklej tu
MODEL          = 'gpt-5.5'
CANONICAL_JSON = 'extended_zurada_canonical_mappings.json'
CHECKPOINT_CSV = 'zurada_descriptions_checkpoint.csv'
OUTPUT_CSV     = 'zurada_canonical_descriptions.csv'
BATCH_SIZE     = 20
SLEEP_BETWEEN  = 0.4
MAX_RETRIES    = 3

client = OpenAI(api_key=OPENAI_API_KEY)
print(f'Model: {MODEL}')


Model: gpt-5.5


In [6]:
with open(CANONICAL_JSON, encoding='utf-8') as f:
    mappings = json.load(f)

COL_CONTEXT = {
    'dozwolone_powierzchnie': 'powierzchnia lub materiał, na którym produkt czyszczący może być bezpiecznie stosowany',
    'odradzane_powierzchnie': 'powierzchnia lub materiał, na którym NIE należy stosować produktu — może go uszkodzić',
    'metoda_mycia':           'metoda lub technika aplikacji/użycia produktu czyszczącego',
    'zgodnosc_i_certyfikaty': 'certyfikat, norma lub standard jakości/bezpieczeństwa związany z produktem',
    'ostrzezenia_bhp':        'ostrzeżenie lub zalecenie bezpieczeństwa przy stosowaniu produktu',
    'odczyn_ph':              'poziom odczynu pH produktu czyszczącego',
}

records = []
seen = set()
for col, info in mappings.items():
    for value in info['canonical_values']:
        key = (value, col)
        if key not in seen:
            seen.add(key)
            records.append({'value': value, 'column': col})

print(f'Lacznie {len(records)} wpisow do opisania ({len(set(r["value"] for r in records))} unikalnych wartosci).')
pd.DataFrame(records).groupby('column').size().rename('count')


Lacznie 462 wpisow do opisania (451 unikalnych wartosci).


column
dozwolone_powierzchnie    307
metoda_mycia               22
odczyn_ph                   5
odradzane_powierzchnie     44
ostrzezenia_bhp            60
zgodnosc_i_certyfikaty     24
Name: count, dtype: int64

In [7]:
SYSTEM_PROMPT = (
    'Jestes ekspertem ds. chemii czyszczecej i higieny przemyslowej.\n'
    'Tworzysz slownik pojec dla systemu rekomendacji produktow czyszczacych.\n\n'
    'Dla kazdej podanej wartosci napisz KROTKI opis (1-2 zdania) po polsku.\n'
    'Opis powinien:\n'
    '- Wyjasniać CO TO JEST w kontekscie chemii czyszczecej / higieny\n'
    '- Byc zrozumialy dla osoby zamawiajacaej srodki czystosci (nie chemika)\n'
    '- Dla skrotow (HACCP, CIP, RSPO, IFRA, EU Ecolabel) — wyjasnic co oznacza skrot i dlaczego jest wazny\n'
    '- Dla powierzchni — opisac material i czemu wymaga uwagi przy czyszczeniu\n'
    '- Dla metod — opisac technikę aplikacji\n'
    '- Dla BHP — opisac na czym polega ryzyko lub zalecenie\n'
    '- NIE zaczynac od "To jest" — pisac wprost\n\n'
    'Odpowiadasz wylacznie JSON: {"descriptions": {"wartosc": "opis", ...}}'
)


def describe_batch(batch):
    lines = []
    for item in batch:
        ctx = COL_CONTEXT.get(item['column'], item['column'])
        lines.append(f'- "{item["value"]}" (kontekst: {ctx})')

    user_msg = (
        f'Opisz ponizsze {len(batch)} pojec z katalogu produktow czyszczacych:\n\n'
        + '\n'.join(lines)
        + '\n\nZwroc JSON: {"descriptions": {"wartosc": "opis", ...}}'
    )

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': user_msg},
                ],
                response_format={'type': 'json_object'},
                max_completion_tokens=2000,
            )
            content = response.choices[0].message.content
            if not content:
                raise ValueError(f'Pusta odpowiedz, refusal={getattr(response.choices[0].message, "refusal", None)}')
            return json.loads(content).get('descriptions', {})
        except Exception as e:
            wait = 2 ** attempt
            print(f'  [RETRY {attempt}/{MAX_RETRIES}] — {e} — czekam {wait}s')
            time.sleep(wait)

    return {}


print('Funkcja describe_batch gotowa.')


Funkcja describe_batch gotowa.


In [8]:
if os.path.exists(CHECKPOINT_CSV):
    done_df   = pd.read_csv(CHECKPOINT_CSV)
    done_keys = set(zip(done_df['value'], done_df['column']))
    print(f'Checkpoint: {len(done_df)} opisow juz gotowych.')
else:
    done_df   = pd.DataFrame(columns=['value', 'column', 'description'])
    done_keys = set()
    print('Brak checkpointu — zaczynam od zera.')

todo = [r for r in records if (r['value'], r['column']) not in done_keys]
print(f'Do opisania: {len(todo)} wpisow w {-(-len(todo)//BATCH_SIZE)} batchach.')


Brak checkpointu — zaczynam od zera.
Do opisania: 462 wpisow w 24 batchach.


In [9]:
new_rows = []
batches  = [todo[i:i+BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]

for batch in tqdm(batches, desc='Generowanie opisow'):
    first_val = batch[0]['value']
    print(f'  batch: {first_val!r} ... ({len(batch)} wpisow)')

    descriptions = describe_batch(batch)

    for item in batch:
        desc = descriptions.get(item['value'], '')
        new_rows.append({'value': item['value'], 'column': item['column'], 'description': desc})

    combined = pd.concat([done_df, pd.DataFrame(new_rows)], ignore_index=True)
    combined.to_csv(CHECKPOINT_CSV, index=False)

    time.sleep(SLEEP_BETWEEN)

print(f'Gotowe. Lacznie {len(done_df) + len(new_rows)} opisow.')


Generowanie opisow:   0%|          | 0/24 [00:00<?, ?it/s]

  batch: 'wszystkie powierzchnie' ... (20 wpisow)
  batch: 'powierzchnie kuchenne' ... (20 wpisow)
  batch: 'sufity' ... (20 wpisow)
  batch: 'kamień sztuczny' ... (20 wpisow)
  batch: 'skóra' ... (20 wpisow)
  batch: 'lastryko' ... (20 wpisow)
  batch: 'maszyny' ... (20 wpisow)
  batch: 'naleśnikarki' ... (20 wpisow)
  batch: 'tkaniny' ... (20 wpisow)
  batch: 'wyparzarki' ... (20 wpisow)
  batch: 'filtry spalin' ... (20 wpisow)
  batch: 'osprzęt silnika' ... (20 wpisow)
  batch: 'elementy gwintowane' ... (20 wpisow)
  batch: 'telewizory' ... (20 wpisow)
  batch: 'elementy chromowane' ... (20 wpisow)
  batch: 'lusterka' ... (20 wpisow)
  batch: 'powierzchnie chłonne' ... (20 wpisow)
  batch: 'powierzchnie ocynkowane' ... (20 wpisow)
  batch: 'mycie cip' ... (20 wpisow)
  batch: 'bez sls' ... (20 wpisow)
  batch: 'aplikować na lakier' ... (20 wpisow)
  batch: 'płukać oczy wodą' ... (20 wpisow)
  batch: 'temperatura wody' ... (20 wpisow)
  batch: 'zasadowy' ... (2 wpisow)
Gotowe. Laczni

In [10]:
final = pd.read_csv(CHECKPOINT_CSV)

empty_mask = final['description'].isna() | (final['description'].str.strip() == '')
print(f'Puste opisy: {empty_mask.sum()} wpisow')
final.loc[empty_mask, 'description'] = final.loc[empty_mask, 'value']

final = final.sort_values(['column', 'value']).reset_index(drop=True)
final.to_csv(OUTPUT_CSV, index=False)
print(f'Zapisano {len(final)} wpisow do {OUTPUT_CSV!r}')
final.head(10)


Puste opisy: 0 wpisow
Zapisano 462 wpisow do 'zurada_canonical_descriptions.csv'


,value,column,description
0,akcesoria kuchenne,dozwolone_powierzchnie,"Drobne wyposażenie kuchni, np. deski, pojemnik..."
1,akcesoria skórzane,dozwolone_powierzchnie,Przedmioty ze skóry naturalnej lub syntetyczne...
2,aluminium,dozwolone_powierzchnie,"Lekki metal używany w elementach urządzeń, oka..."
3,aluminium polerowane,dozwolone_powierzchnie,"Gładko wykończone aluminium o błyszczącej, dek..."
4,armatura,dozwolone_powierzchnie,"Baterie, krany, słuchawki prysznicowe i inne e..."
5,autobusy,dozwolone_powierzchnie,Pojazdy komunikacji zbiorowej z dużymi powierz...
6,balustrady,dozwolone_powierzchnie,"Balustrady wykonuje się ze stali, aluminium, s..."
7,bemary,dozwolone_powierzchnie,Podgrzewacze gastronomiczne do utrzymywania te...
8,beton,dozwolone_powierzchnie,Porowaty materiał mineralny stosowany na posad...
9,bez kontaktu z żywnością,dozwolone_powierzchnie,"Powierzchnie lub zastosowania, w których czysz..."


In [11]:
print('=== Certyfikaty i zgodnosci ===')
cert_df = final[final['column'] == 'zgodnosc_i_certyfikaty']
for _, row in cert_df.iterrows():
    print(f'\n[{row["value"]}]')
    print(f'  {row["description"]}')


=== Certyfikaty i zgodnosci ===

[bez dodecylobenzenosulfonianów]
  Formuła bez dodecylobenzenosulfonianów, czyli detergentów anionowych stosowanych do odtłuszczania i piany. Taka informacja bywa istotna przy wyborze łagodniejszych lub bardziej środowiskowo ukierunkowanych środków czyszczących.

[bez edta]
  Formuła bez EDTA, czyli związku wiążącego jony metali i wspomagającego działanie detergentów w twardej wodzie. Ważne dla użytkowników szukających preparatów o mniejszym obciążeniu środowiska.

[bez eutrofizacji]
  Deklaracja oznacza, że produkt został opracowany tak, aby ograniczać ryzyko wzbogacania wód w składniki odżywcze powodujące zakwity glonów. Ma znaczenie środowiskowe, szczególnie przy detergentach spłukiwanych do kanalizacji.

[bez fosforanów]
  Oznaczenie informuje, że produkt nie zawiera fosforanów stosowanych dawniej do zmiękczania wody i wzmacniania prania lub mycia. Jest ważne, ponieważ fosforany mogą przyczyniać się do zanieczyszczenia wód i nadmiernego rozwoju glon

In [12]:
print('=== Metody mycia ===')
for _, row in final[final['column'] == 'metoda_mycia'].iterrows():
    print(f'\n[{row["value"]}]')
    print(f'  {row["description"]}')


=== Metody mycia ===

[bezdotykowe]
  Metoda polega na nanoszeniu środka czyszczącego i spłukiwaniu go wodą pod ciśnieniem bez mechanicznego pocierania. Jest popularna w myjniach i ogranicza ryzyko zarysowania czyszczonej powierzchni.

[ciśnieniowe]
  Czyszczenie z użyciem wody lub roztworu środka pod podwyższonym ciśnieniem, najczęściej myjką ciśnieniową. Metoda skutecznie usuwa luźne i uporczywe zabrudzenia, ale może uszkodzić delikatne powierzchnie lub uszczelnienia.

[czyszczenie]
  Ogólna metoda usuwania brudu, tłuszczu, osadów lub kurzu z powierzchni przy użyciu odpowiedniego preparatu. Może wymagać spryskania, przetarcia, szorowania lub pozostawienia środka na określony czas.

[ekstrakcyjne]
  Czyszczenie ekstrakcyjne polega na natrysku roztworu czyszczącego w tkaninę i jednoczesnym odsysaniu brudnej wody. Stosuje się je do dywanów, wykładzin i tapicerki, gdzie ważne jest usunięcie zabrudzeń z głębszych warstw.

[maszynowe]
  Użycie produktu w urządzeniach czyszczących, takich j

In [13]:
SEARCH = 'CIP'
mask = (
    final['value'].str.contains(SEARCH, case=False, na=False)
    | final['description'].str.contains(SEARCH, case=False, na=False)
)
final[mask][['value', 'column', 'description']]


,value,column,description
312,mycie cip,metoda_mycia,Mycie CIP (Cleaning in Place) polega na czyszc...
